# Événements de pluie et réponse hydrogéomorphologique
Le pont couvert de la rivière Neigette a été remplacé le **12 septembre 2017**. L'objectif de cette partie est de vous assister dans la production de la présentation finale. Pour y arriver, voici les étapes que j'avais en tête:
1. Délimiter les événements de pluie dans les données météorologiques.
2. Identifier les événements qui dépassent un seuil de déclenchement.
3. Évaluer si les conditions hydroclimatiques sont comparables avant et après le changement de pont pour que vous puissiez approfondir et nuancer vos analyses en ce qui a trait à la migration de la rivière.


**Démarche pour chaque bloc**: (1) définir les paramètres dans la cellule **PARAMÈTRES**, (2) lire les figures et le tableau de sensibilité et (3) préparez vous à l'évaluation à l'aide de la liste de questions fournies.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from outils import lire, delimiter, seuil, annee_pont, verifier
pap, ne = lire("PaP"), lire("Neigette")
p, T = pap.precip_mm["2004-09-12":"2026-09-11 23:00"], pap.temp_air_C

## Bloc 1. Délimitation des événements de pluie
Les événements de précipitation sont délimités par *SEC* heures sans pluie. Une heure isolée ayant une valeur de précipitations de 0.1 mm peut artificiellement prolonger un événement et atténuer son intensité: les heures ayant des valeurs inférieures ou égales à PMIN sont donc traitées comme nulles pour la délimitation des événements, mais pas pour les calculs.

Le seuil intensité–durée qui sera utilisé s'intéresse à la pluie, pas la neige. Le paramètre T_PLUIE sert à définir à partir de quelle température considérer les précipitations comme étant liquides.

> **ATTENTION**: définissez les paramètres de la cellule suivante, puis exécutez-la.

In [ ]:
# PARAMÈTRES
SEC = None       # heures sans pluie séparant deux événements : 6, 12 ou 24
PMIN = None      # heures ≤ PMIN (mm) considérées sèches : 0, 0.1, 0.2 ou 0.5
T_PLUIE = None   # température moyenne minimale (°C) d'un événement de pluie : 0, 1, 2 ou 3
verifier(SEC=SEC, PMIN=PMIN, T_PLUIE=T_PLUIE)

def evenements(sec, pmin, t_pluie):
    E = delimiter(p, T, mit=sec, pmin=pmin)
    return E[E.T_moy > t_pluie]

E = evenements(SEC, PMIN, T_PLUIE)
E[["D", "P", "I", "Imax"]].describe().round(1)

In [ ]:
from IPython.display import display
colonnes = ["SEC (h)", "PMIN (mm)", "T_PLUIE (°C)", "événements", "durée médiane (h)", "durée max (h)", "dépassements du seuil"]
def ligne(sec, pmin, t):
    e = evenements(sec, pmin, t)
    return [sec, pmin, t, len(e), e.D.median(), e.D.max(), e.depasse.sum()]
print(f"Effet de SEC et PMIN (T_PLUIE = {T_PLUIE} °C)")
display(pd.DataFrame([ligne(sec, pmin, T_PLUIE) for sec in [6, 12, 24] for pmin in [0, 0.2, 0.5]], columns=colonnes))
print(f"Effet de T_PLUIE (SEC = {SEC} h, PMIN = {PMIN} mm)")
display(pd.DataFrame([ligne(SEC, PMIN, t) for t in [0, 1, 2, 3]], columns=colonnes))

## Bloc 2. Événements dépassant un seuil de déclenchement d'aléas
Seuil de déclenchement d'aléas hydrogéomorphologiques pour le nord de la Gaspésie (Boulet et al., 2022) : **I = 10,5 · D^-0,58**.
Un événement positionné au-dessus du seuil a le pootentiel d'avoir eu des impacts morphologiques sur la rivière (érosion, apport sédimentaire, etc.).

In [ ]:
D = np.logspace(0, 2.3)
plt.figure(figsize=(7, 5))
plt.scatter(E.D, E.I, s=10, c="lightgrey", label="événements de pluie")
plt.scatter(E[E.depasse].D, E[E.depasse].I, s=25, facecolors="none", edgecolors="k", label="> seuil")
plt.plot(D, seuil(D), "darkred", label="Boulet et al. (2022)")
plt.xscale("log"); plt.yscale("log"); plt.xlabel("Durée (h)"); plt.ylabel("Intensité moyenne (mm/h)"); plt.legend();
print("99,9e centile des heures pluvieuses :", p[p > 0].quantile(.999).round(1), "mm")
E[E.depasse].assign(ratio=lambda d: d.I / seuil(d.D)).nlargest(6, "ratio")[["debut", "D", "P", "I", "Imax", "ratio"]].round(1)

In [ ]:
pap.loc["2005-10-07 16:00":"2005-10-08 00:00", ["precip_mm", "hr_pct", "temp_air_C"]].T

> **ATTENTION** : définissez les paramètres de la cellule suivante, puis exécutez-la.

In [ ]:
# PARAMÈTRES
IMAX_PLAUSIBLE = None   # pluie horaire maximale jugée plausible (mm) : nombre entier ou "aucun" (tout conserver)
verifier(IMAX_PLAUSIBLE=IMAX_PLAUSIBLE)

def nettoyer(E, imax):
    return (E if imax == "aucun" else E[E.Imax < imax]).copy()

Ep = nettoyer(E, IMAX_PLAUSIBLE)
print(f"{len(E) - len(Ep)} événements retirés ; {Ep.depasse.sum()} dépassements retenus sur {len(Ep)} événements")

In [ ]:
pd.DataFrame([[v, len(E) - len(nettoyer(E, v)), nettoyer(E, v).depasse.sum(),
               ", ".join(E[~E.index.isin(nettoyer(E, v).index)].nlargest(4, "Imax").Imax.round(1).astype(str))]
              for v in [30, 50, 100, "aucun"]],
             columns=["IMAX_PLAUSIBLE", "événements retirés", "dépassements retenus", "Imax retirés (mm, 4 plus grands)"])

## Bloc 3. Comparaison avant et après le changement d'infrastructure

> **ATTENTION** : définissez les paramètres de la cellule suivante, puis exécutez-la.

In [ ]:
# PARAMÈTRES
DEBUT_REFERENCE = None   # Année du début de la période « avant » : nombre entier
TEST_TAUX = None         # "poisson" (comparaison de deux taux) ou "mann-whitney" (comptes annuels)
verifier(DEBUT_REFERENCE=DEBUT_REFERENCE, TEST_TAUX=TEST_TAUX)

Ep["annee"] = annee_pont(Ep.debut)
n_an = Ep[Ep.depasse].groupby("annee").size().reindex(range(2004, 2026), fill_value=0)

def comparer(debut_ref, test):
    n = n_an[n_an.index >= debut_ref]; av, ap = n[n.index < 2017], n[n.index >= 2017]
    pval = (stats.binomtest(int(ap.sum()), int(n.sum()), len(ap) / len(n)).pvalue if test == "poisson"
            else stats.mannwhitneyu(av, ap).pvalue)
    return {"années avant": len(av), "taux avant (/an)": round(av.mean(), 2), "taux après (/an)": round(ap.mean(), 2),
            "rapport après/avant": round(ap.mean() / av.mean(), 2), "p": round(pval, 2)}

n = n_an[n_an.index >= DEBUT_REFERENCE]
plt.bar(n.index, n.values, color=np.where(n.index < 2017, "tab:blue", "tab:orange")); plt.ylabel("dépassements / an")
plt.title("Bleu : avant le pont ; orange : après");
print("Prémisses — autocorrélation des comptes :", round(n.autocorr(1), 2), "; variance/moyenne :", round(n.var() / n.mean(), 2))
pd.Series(comparer(DEBUT_REFERENCE, TEST_TAUX))

In [ ]:
pd.DataFrame({f"{a} / {t}": comparer(a, t) for a in [2004, 2008, 2013] for t in ["poisson", "mann-whitney"]}).T

### 3b. Bilan utilisant des intervalles de dates pour comparer avant et après

> **ATTENTION** : définissez les intervalles de la cellule suivante, puis exécutez-la.

In [ ]:
# PARAMÈTRES
# Autant d'intervalles que vous voulez : {"nom": ("début AAAA-MM-JJ", "fin AAAA-MM-JJ"), ...}
# Dates possibles : du 2004-09-12 au 2026-09-11.
# Suggestion : les dates des images de votre projet (LiDAR 2013-10-29 et 2019-10-27 ; orthophotos 2015-09-21 et 2022-05-20)
INTERVALLES = None   # p. ex. {"2013 à 2015": ("2013-10-29", "2015-09-21"), "2019 à 2022": ("2019-10-27", "2022-05-20")}
verifier(INTERVALLES=INTERVALLES)
PONT = pd.Timestamp("2017-09-12")

def bilan(debut, fin):
    e, q = Ep[(Ep.debut >= debut) & (Ep.debut < fin)], ne.debit_m3s[debut:fin]
    ans = (pd.Timestamp(fin) - pd.Timestamp(debut)).days / 365.25
    position = "avant le pont" if pd.Timestamp(fin) <= PONT else "après le pont" if pd.Timestamp(debut) >= PONT else "chevauche le pont"
    return {"période": position, "années": round(ans, 1), "dépassements": int(e.depasse.sum()), "dépassements/an": round(e.depasse.sum() / ans, 1),
            "pluie (mm/an)": round(p[debut:fin].sum() / ans), "débit max (m³/s)": q.max(), "débit manquant (%)": round(q.isna().mean() * 100)}
pd.DataFrame({k: bilan(*v) for k, v in INTERVALLES.items()}).T